# Transfer DICOM series to NIfTI by split


In [ ]:
import json
import os
import shutil
import dicom2nifti

DATA_SPLIT = "train"  # options: "train", "val", "test", "newcase"

SPLIT_CONFIG = {
    "train": {
        "dataset_dir": r"/path/to/datasets/DATASET_UUID_REDACTED",
        "output_dir": r"/path/to/workspace/train_dataset_nifti",
        "study_limit": None,
    },
    "val": {
        "dataset_dir": r"/path/to/datasets/DATASET_UUID_REDACTED",
        "output_dir": r"/path/to/workspace/val_dataset_nifti",
        "study_limit": None,
    },
    "test": {
        "dataset_dir": r"/path/to/datasets/DATASET_UUID_REDACTED",
        "output_dir": r"/path/to/workspace/test_dataset_nifti",
        "study_limit": None,
    },
    "newcase": {
        "dataset_dir": r"/path/to/datasets/DATASET_UUID_REDACTED",
        "output_dir": r"/path/to/workspace/Newcase_dataset_nifti",
        "study_limit": None,
    },
}

config = SPLIT_CONFIG[DATA_SPLIT]
dataset_dir = config["dataset_dir"]
output_dir = config["output_dir"]
file_name_harmonized = "harmonization_sample.nii.gz"


In [ ]:
index_file_path = os.path.join(dataset_dir, "index.json")
with open(index_file_path) as f:
    studies_index = json.load(f)

if config["study_limit"] is not None:
    studies_index = studies_index[: config["study_limit"]]

print(DATA_SPLIT, "studies:", len(studies_index))


In [ ]:
for study in studies_index:
    series_list = study["series"]
    studies_path = study["path"]
    studies_path_all = os.path.join(dataset_dir, studies_path)
    output_path_all = os.path.join(output_dir, studies_path)
    os.makedirs(output_path_all, exist_ok=True)

    for series in series_list:
        series_folder = series["folderName"]
        series_directory = os.path.join(studies_path_all, series_folder)

        if series["tags"] == []:
            series_folder_new = series_folder[-4:]
            output_folder = os.path.join(output_path_all, series_folder_new)
            os.makedirs(output_folder, exist_ok=True)
            dicom2nifti.convert_directory(series_directory, output_folder, compression=True, reorient=True)

        elif series["tags"] == ["HARMONIZED"]:
            new_file_name = series_folder.split("-")[0][-4:]
            output_folder = os.path.join(output_path_all, new_file_name)
            os.makedirs(output_folder, exist_ok=True)
            src_path = os.path.join(series_directory, file_name_harmonized)
            shutil.copy(src_path, output_folder)

print("done")
